In [16]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD Python module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "********"

# Connect to database via CRUD Module
shelter = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare's logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Header section with logo and title
    html.Center([
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '100px', 'margin': '20px'}),
        html.H1('Grazioso Salvare Animal Shelter Dashboard',
               style={'color': '#C41E3A', 'font-weight': 'bold'}),
        html.H3('Sam Herbst - CS 340 Project Two',
               style={'color': '#555', 'font-style': 'italic'})
    ]),
    
    html.Hr(),
    
    # Interactive filter options section
    html.Div([
        html.H4('Filter Options:', style={'margin-bottom': '10px'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' Reset (Show All)', 'value': 'reset'},
                {'label': ' Water Rescue', 'value': 'water'},
                {'label': ' Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': ' Disaster/Individual Tracking', 'value': 'disaster'}
            ],
            value='reset',
            inline=True,
            labelStyle={'margin-right': '20px', 'font-size': '16px'}
        )
    ], style={'margin': '20px', 'padding': '15px', 'background-color': '#f0f0f0', 
              'border-radius': '5px'}),
    
    html.Hr(),
    
    # Data table
    html.Div([
        html.H4('Available Animals:', style={'margin-bottom': '10px'}),
        dash_table.DataTable(
            id='datatable-id',
            columns=[{"name": i, "id": i, "deletable": False, "selectable": True} 
                    for i in df.columns],
            data=df.to_dict('records'),
            editable=False,
            filter_action="native",
            sort_action="native",
            sort_mode="multi",
            column_selectable=False,
            row_selectable="single",
            row_deletable=False,
            selected_columns=[],
            selected_rows=[0],
            page_action="native",
            page_current=0,
            page_size=10,
            style_table={'overflowX': 'auto'},
            style_cell={
                'textAlign': 'left',
                'padding': '10px',
                'font-family': 'Arial'
            },
            style_header={
                'backgroundColor': '#C41E3A',
                'color': 'white',
                'fontWeight': 'bold'
            },
            style_data_conditional=[
                {
                    'if': {'row_index': 'odd'},
                    'backgroundColor': '#f9f9f9'
                }
            ]
        )
    ], style={'margin': '20px'}),
    
    html.Br(),
    html.Hr(),
    
    # Charts section - side by side
    html.Div(className='row',
        style={'display': 'flex'},
        children=[
            # Pie chart for breed distribution
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%', 'padding': '10px'}
            ),
            # Geolocation map
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%', 'padding': '10px'}
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Callback to update data table based on filter selection
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """
    Filter the data table based on rescue type selection.
    Uses queries that match the requirements from the Dashboard Specifications.
    """
    if filter_type == 'water':
        # Water Rescue: Labrador Retriever Mix, Chesapeake Bay Retriever, Newfoundland
        # Intact Female, 26-156 weeks old
        query = {
            'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'mountain':
        # Mountain/Wilderness Rescue: German Shepherd, Alaskan Malamute, 
        # Old English Sheepdog, Siberian Husky, Rottweiler
        # Intact Male, 26-156 weeks old
        query = {
            'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 
                             'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'disaster':
        # Disaster/Individual Tracking: Doberman Pinscher, German Shepherd,
        # Golden Retriever, Bloodhound, Rottweiler
        # Intact Male, 20-300 weeks old
        query = {
            'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 
                             'Golden Retriever', 'Bloodhound', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }
    else:  # reset
        # Show all animals
        query = {}
    
    # Query the database using CRUD module
    df_filtered = pd.DataFrame.from_records(shelter.read(query))
    
    if not df_filtered.empty:
        df_filtered.drop(columns=['_id'], inplace=True)
    
    return df_filtered.to_dict('records')

# Callback to update pie chart based on filtered data
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    """
    Display breed distribution as a pie chart based on the current data table view.
    """
    if viewData is None or len(viewData) == 0:
        return [html.Div("No data available for chart")]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Count breeds and create pie chart
    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    
    # Limit to top 10 breeds for readability
    breed_counts = breed_counts.head(10)
    
    return [
        dcc.Graph(
            figure=px.pie(
                breed_counts,
                values='count',
                names='breed',
                title='Top 10 Breed Distribution',
                color_discrete_sequence=px.colors.sequential.RdBu
            )
        )
    ]

# Callback to highlight selected columns in data table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    """Highlight selected columns in the data table."""
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# Callback to update geolocation map based on selected row
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    """
    Update the geolocation map to show the selected animal's location.
    """
    if viewData is None or len(viewData) == 0:
        return [html.Div("No data available for map")]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Use first row if no selection
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]
    
    # Check if row exists in current view
    if row >= len(dff):
        row = 0
    
    # Get location data (handling potential missing values)
    lat = dff.iloc[row].get('location_lat', 30.75)
    lon = dff.iloc[row].get('location_long', -97.48)
    breed = dff.iloc[row].get('breed', 'Unknown')
    name = dff.iloc[row].get('name', 'Unknown')
    
    # Austin TX coordinates as center
    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                # Marker with tooltip and popup
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([
                            html.H4("Animal Details"),
                            html.P(f"Name: {name}"),
                            html.P(f"Breed: {breed}"),
                            html.P(f"Location: ({lat:.4f}, {lon:.4f})")
                        ])
                    ]
                )
            ]
        )
    ]

# Run the app
app.run_server(debug=True)

Dash app running on https://saharadynasty-editionspoon-3000.codio.io/proxy/8050/
